# PDF File Tracking System

This notebook implements a system to scan a directory recursively for PDF files and track their existence in a PostgreSQL database. The system will:
1. Scan for PDFs with format: `{file_type}_{agent}_{policy_num}.pdf`
2. Track which file types exist for each policy number
3. Store last modification dates for each file
4. Provide summary reports of file existence

## Import Required Libraries

In [ ]:
import os
from pathlib import Path
import psycopg2
from psycopg2 import sql
from datetime import datetime
import pandas as pd
import re

## Configure Database Connection

In [ ]:
# Database connection parameters
DB_PARAMS = {
    'dbname': 'your_database',
    'user': 'your_user',
    'password': 'your_password',
    'host': 'localhost',
    'port': '5432'
}

def get_db_connection():
    """Create and return a database connection."""
    try:
        conn = psycopg2.connect(**DB_PARAMS)
        return conn
    except psycopg2.Error as e:
        print(f"Error connecting to the database: {e}")
        raise

## Create Database Schema

In [ ]:
def create_tables():
    """Create the necessary database tables if they don't exist."""
    conn = get_db_connection()
    cur = conn.cursor()
    
    try:
        # Create policy_manager_file_types table
        cur.execute("""
            CREATE TABLE IF NOT EXISTS policy_manager_file_types (
                id SERIAL PRIMARY KEY,
                type_name VARCHAR(100) UNIQUE NOT NULL
            );
        """)
        
        # Create policy_manager_policies table
        cur.execute("""
            CREATE TABLE IF NOT EXISTS policy_manager_policies (
                id SERIAL PRIMARY KEY,
                policy_num VARCHAR(100) UNIQUE NOT NULL,
                agent VARCHAR(100) NOT NULL
            );
        """)
        
        # Create policy_manager_file_status table
        cur.execute("""
            CREATE TABLE IF NOT EXISTS policy_manager_file_status (
                id SERIAL PRIMARY KEY,
                policy_id INTEGER REFERENCES policy_manager_policies(id),
                file_type_id INTEGER REFERENCES policy_manager_file_types(id),
                last_modified TIMESTAMP NOT NULL,
                file_path TEXT NOT NULL,
                UNIQUE(policy_id, file_type_id)
            );
        """)
        
        conn.commit()
        print("Tables created successfully")
        
    except psycopg2.Error as e:
        print(f"Error creating tables: {e}")
        conn.rollback()
        raise
    finally:
        cur.close()
        conn.close()

# Create tables
create_tables()

## Scan Directory for PDFs

In [ ]:
def scan_directory(root_path):
    """Recursively scan directory for PDF files."""
    pdf_files = []
    root = Path(root_path)
    
    for path in root.rglob('*.pdf'):
        pdf_files.append({
            'path': str(path),
            'modified_time': datetime.fromtimestamp(path.stat().st_mtime)
        })
    
    return pdf_files

# Example usage:
# pdf_files = scan_directory('/path/to/your/directory')

## Parse PDF Filenames

In [ ]:
def parse_filename(file_path):
    """Parse PDF filename to extract file_type, agent, and policy_num."""
    filename = Path(file_path).stem  # Get filename without extension
    pattern = r'^(.+?)_(.+?)_(.+)$'
    
    match = re.match(pattern, filename)
    if not match:
        return None
    
    file_type, agent, policy_num = match.groups()
    return {
        'file_type': file_type,
        'agent': agent,
        'policy_num': policy_num
    }

def process_pdf_files(pdf_files):
    """Process list of PDF files and extract information."""
    processed_files = []
    
    for pdf_file in pdf_files:
        file_info = parse_filename(pdf_file['path'])
        if file_info:
            file_info.update({
                'file_path': pdf_file['path'],
                'modified_time': pdf_file['modified_time']
            })
            processed_files.append(file_info)
    
    return processed_files

## Update Database Tables

In [ ]:
def update_database(processed_files):
    """Update database with processed file information."""
    conn = get_db_connection()
    cur = conn.cursor()
    
    try:
        # First, collect all unique file types and policies
        file_types = set()
        policies = {}  # policy_num -> agent
        
        for file_info in processed_files:
            file_types.add(file_info['file_type'])
            policies[file_info['policy_num']] = file_info['agent']
        
        # Insert file types
        file_type_ids = {}
        for file_type in file_types:
            cur.execute("""
                INSERT INTO policy_manager_file_types (type_name)
                VALUES (%s)
                ON CONFLICT (type_name) DO UPDATE
                SET type_name = EXCLUDED.type_name
                RETURNING id, type_name;
            """, (file_type,))
            id_, type_name = cur.fetchone()
            file_type_ids[type_name] = id_
        
        # Insert policies
        policy_ids = {}
        for policy_num, agent in policies.items():
            cur.execute("""
                INSERT INTO policy_manager_policies (policy_num, agent)
                VALUES (%s, %s)
                ON CONFLICT (policy_num) DO UPDATE
                SET agent = EXCLUDED.agent
                RETURNING id, policy_num;
            """, (policy_num, agent))
            id_, policy_num = cur.fetchone()
            policy_ids[policy_num] = id_
        
        # Update file status
        for file_info in processed_files:
            cur.execute("""
                INSERT INTO policy_manager_file_status (policy_id, file_type_id, last_modified, file_path)
                VALUES (%s, %s, %s, %s)
                ON CONFLICT (policy_id, file_type_id) DO UPDATE
                SET last_modified = EXCLUDED.last_modified,
                    file_path = EXCLUDED.file_path;
            """, (
                policy_ids[file_info['policy_num']],
                file_type_ids[file_info['file_type']],
                file_info['modified_time'],
                file_info['file_path']
            ))
        
        conn.commit()
        print("Database updated successfully")
        
    except psycopg2.Error as e:
        print(f"Error updating database: {e}")
        conn.rollback()
        raise
    finally:
        cur.close()
        conn.close()

## Query and Report Results

In [ ]:
def get_file_status_report():
    """Generate a report of file existence for each policy."""
    conn = get_db_connection()
    
    try:
        query = """
            WITH file_matrix AS (
                SELECT 
                    p.policy_num,
                    p.agent,
                    ft.type_name,
                    fs.last_modified,
                    fs.file_path
                FROM policy_manager_policies p
                CROSS JOIN policy_manager_file_types ft
                LEFT JOIN policy_manager_file_status fs ON fs.policy_id = p.id AND fs.file_type_id = ft.id
            )
            SELECT * FROM file_matrix
            ORDER BY policy_num, type_name;
        """
        
        df = pd.read_sql_query(query, conn)
        
        # Pivot the data to show file types as columns
        pivot_df = df.pivot_table(
            index=['policy_num', 'agent'],
            columns='type_name',
            values='last_modified',
            aggfunc='first'
        ).reset_index()
        
        return pivot_df
        
    except psycopg2.Error as e:
        print(f"Error generating report: {e}")
        raise
    finally:
        conn.close()

# Example usage of the complete system
def main(root_path):
    """Run the complete file tracking system."""
    try:
        # Scan directory for PDF files
        pdf_files = scan_directory(root_path)
        print(f"Found {len(pdf_files)} PDF files")
        
        # Process the files
        processed_files = process_pdf_files(pdf_files)
        print(f"Successfully processed {len(processed_files)} files")
        
        # Update database
        update_database(processed_files)
        
        # Generate and display report
        # report = get_file_status_report()
        # print("\nFile Status Report:")
        # print(report)
        
        # return report
        
    except Exception as e:
        print(f"Error in main process: {e}")
        raise

# Example execution:
# main('/path/to/your/directory')